In [ ]:
!pip install -q groq pandas sentence-transformers

In [ ]:
import os
import re
import json
import time
import random
import threading
import urllib.request
import difflib
from collections import defaultdict

import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Creating the folder for storing extraction and verification files
DRIVE_DIR = "/content/drive/MyDrive/apt_extraction"
os.makedirs(DRIVE_DIR, exist_ok=True)

# Defining paths for candidate, progress and verification files
CANDIDATES_PATH = os.path.join(DRIVE_DIR, "raw_llm_candidates.json")
PROGRESS_PATH = os.path.join(DRIVE_DIR, "verification_progress.json")
VERIFIED_PATH = os.path.join(DRIVE_DIR, "groq_verified_full_log.json")
ATTACK_PSEUDO_PATH = os.path.join(DRIVE_DIR, "verified_pseudo_labels.json")
COSINE_PSEUDO_PATH = os.path.join(DRIVE_DIR, "verified_pseudo_labels_cosine.json")

# Loading the extracted candidate triples
raw_candidates_df = pd.read_json(CANDIDATES_PATH)

# Printing the number of candidate triples loaded for verification
print(f"Loaded {len(raw_candidates_df)} candidate triples for verification.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 3979 candidate triples for verification.


## MITRE ATT&CK grounding

In [ ]:
# Normalizing text by converting it to lowercase and removing spaces, dashes and underscores
def _norm(s: str) -> str:
    return s.lower().replace(" ", "").replace("-", "").replace("_", "")


# Defining URLs for MITRE ATT&CK STIX data
ATTACK_STIX_URLS = {
    "enterprise": "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/enterprise-attack/enterprise-attack.json",
    "mobile":     "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/mobile-attack/mobile-attack.json",
    "ics":        "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/ics-attack/ics-attack.json",
}


# Loading MITRE ATT&CK STIX data and cache it locally
def load_attack_stix(domains=("enterprise",), cache_dir="attack_stix"):
    # Creating the cache directory if it does not exist
    os.makedirs(cache_dir, exist_ok=True)
    objects = []
    # Loading STIX data for each selected ATT&CK domain
    for domain in domains:
        url = ATTACK_STIX_URLS[domain]
        local_path = os.path.join(cache_dir, f"{domain}-attack.json")
        # Downloading the STIX file if it is not already cached
        if not os.path.exists(local_path):
            print(f"Downloading {domain} ATT&CK STIX bundle")
            urllib.request.urlretrieve(url, local_path)
        # Reading the STIX bundle from the local file
        with open(local_path, "r", encoding="utf-8") as fh:
            bundle = json.load(fh)
        # Adding the STIX objects to the collection
        objects.extend(bundle["objects"])
        print(f"  {domain}: {len(bundle['objects'])} STIX objects loaded")
    return objects


# Mapping STIX object types to the entity types used in the project
ATTACK_TYPE_MAP = {"attack-pattern": "attack-pattern",
    "malware":        "malware",
    "tools":          "tool",
    "threat-actor":   "intrusion-set",
    "campaign":       "campaign",
}


# Defining the expected format for CVE identifiers
CVE_PATTERN = re.compile(r"^CVE-\d{4}-\d{4,7}$", re.IGNORECASE)
# Building a lookup dictionary of MITRE ATT&CK names and aliases
def build_attack_kb(stix_objects: list) -> dict:
    kb = defaultdict(dict)
    # Processing each STIX object
    for obj in stix_objects:
        obj_type = obj.get("type")
      # Skipping unsupported object types
        if obj_type not in ATTACK_TYPE_MAP.values():
            continue
        # Skipping revoked or deprecated ATT&CK objects
        if obj.get("revoked") or obj.get("x_mitre_deprecated"):
            continue
        # Getting the MITRE ATT&CK external ID
        ext_id = next((r["external_id"] for r in obj.get("external_references", []) if r.get("source_name") == "mitre-attack"), None,)
        # Getting the main name and aliases for the object
        name = obj.get("name", "")
        names = {name, *obj.get("aliases", []), *obj.get("x_mitre_aliases", [])}
        # Adding each name and alias to the knowledge base
        for n in names:
            if n:
                kb[obj_type][_norm(n)] = (name, ext_id)

    return kb


# Loading MITRE ATT&CK reference data for entity verification
print("Loading MITRE ATT&CK STIX reference data for verification")
attack_stix_objects = load_attack_stix(domains=("enterprise",))

# Building the MITRE ATT&CK knowledge base
attack_kb = build_attack_kb(attack_stix_objects)

# Displaying the number of known names and aliases for each entity type
for _t, _d in attack_kb.items():
    print(f"  {_t}: {len(_d)} known names/aliases")

Loading MITRE ATT&CK STIX reference data for verification
  enterprise: 26086 STIX objects loaded
  campaign: 60 known names/aliases
  intrusion-set: 598 known names/aliases
  malware: 987 known names/aliases
  tool: 114 known names/aliases
  attack-pattern: 672 known names/aliases


In [ ]:
# Looking up an entity in the MITRE ATT&CK knowledge base
def attack_kb_lookup(name: str, entity_type: str, cutoff: float = 0.85):
    entity_type_lower = str(entity_type).lower()
    # Checking the format of vulnerability names using the CVE pattern
    if entity_type_lower == "vulnerability":
        if CVE_PATTERN.match(str(name).strip()):
            return True, str(name).strip().upper(), None, 1.0
        return False, None, None, 0.0

    # "SOFTWARE" isn't a distinct STIX type in ATT&CK, it's handled as a special case in attack_kb_lookup, checking both "malware" and "tool" buckets
    # Checking both malware and tool categories for software entities
    if entity_type_lower == "software":
        for stix_type in ("malware", "tool"):
            table = attack_kb.get(stix_type, {})
            norm_name = _norm(name)
            # Checking for an exact entity name match
            if norm_name in table:
                canonical, ext_id = table[norm_name]
                return True, canonical, ext_id, 1.0
            # Checking for a close name match
            match = difflib.get_close_matches(norm_name, table.keys(), n=1, cutoff=cutoff)
            if match:
                canonical, ext_id = table[match[0]]
                score = difflib.SequenceMatcher(None, norm_name, match[0]).ratio()
                return True, canonical, ext_id, round(score, 3)

        return False, None, None, 0.0

    # Mapping the entity type to the corresponding STIX type
    stix_type = ATTACK_TYPE_MAP.get(entity_type)
    if stix_type is None or stix_type not in attack_kb:
        return False, None, None, 0.0
    # Getting the knowledge base entries for the selected entity type
    table = attack_kb[stix_type]
    norm_name = _norm(name)
    # Checking for an exact entity name match
    if norm_name in table:
        canonical, ext_id = table[norm_name]
        return True, canonical, ext_id, 1.0
    # Checking for a close name match when an exact match is not found
    match = difflib.get_close_matches(norm_name, table.keys(), n=1, cutoff=cutoff)
    if match:
        canonical, ext_id = table[match[0]]
        score = difflib.SequenceMatcher(None, norm_name, match[0]).ratio()
        return True, canonical, ext_id, round(score, 3)
    # Returning no match when the entity cannot be found
    return False, None, None, 0.0


# Adding MITRE ATT&CK matching information to the candidate DataFrame
def annotate_with_attack_kb(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    # Creating a copy of the input DataFrame
    df = df.copy()
    # Finding ATT&CK matches for head and tail entities
    head_matches = df.apply(lambda r: attack_kb_lookup(r["head"], r["head_type"]), axis=1)
    tail_matches = df.apply(lambda r: attack_kb_lookup(r["tail"], r["tail_type"]), axis=1)
    # Storing the ATT&CK matching results for head entities
    df["head_attack_match"] = [m[0] for m in head_matches]
    df["head_attack_name"] = [m[1] for m in head_matches]
    df["head_attack_id"] = [m[2] for m in head_matches]
    # Storing the ATT&CK matching results for tail entities
    df["tail_attack_match"] = [m[0] for m in tail_matches]
    df["tail_attack_name"] = [m[1] for m in tail_matches]
    df["tail_attack_id"] = [m[2] for m in tail_matches]

    return df


# Annotating candidate triples with MITRE ATT&CK entity matches
raw_candidates_df = annotate_with_attack_kb(raw_candidates_df)
# Counting and displaying the number of ATT&CK-grounded entity mentions
n_grounded = int(raw_candidates_df["head_attack_match"].sum() + raw_candidates_df["tail_attack_match"].sum()) if not raw_candidates_df.empty else 0
print(f"ATT&CK-grounded entity mentions found in candidates: {n_grounded}")

ATT&CK-grounded entity mentions found in candidates: 1160


In [ ]:
# Loading the Groq API key and initialize the Groq client and setting the LLM model used for verification
from google.colab import userdata
from groq import Groq
client = Groq(api_key=userdata.get("GROQ_API_KEY"))
VERIFIER_MODEL = "openai/gpt-oss-120b"

In [ ]:
import time
import random

# Setting the maximum number of retry attempts
MAX_RETRIES = 5
# Sending a prompt to the Groq API with retry handling
def call_groq(prompt: str, max_tokens: int = 600):
    for attempt in range(MAX_RETRIES):
        try:
            # Adding a short delay between API requests
            time.sleep(2.0)
            return client.chat.completions.create(model=VERIFIER_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=max_tokens,
                reasoning_effort="low",  # gpt-oss defaults to "medium" if unset which eats into max_tokens with hidden reasoning before the JSON even starts
            )

        except Exception as e:
            error = str(e)
            # Checking if the daily token quota has been reached
            if "TPD" in error or "tokens per day" in error.lower():
                print("Groq daily token limit reached. Stopping.")
                return "DAILY_LIMIT_EXCEEDED"
            # Checking for JSON generation or temporary API errors
            json_gen_error = ( "failed to generate json" in error.lower()  or "failed to validate json" in error.lower() )
            retryable = json_gen_error or any( code in error for code in ["429", "500", "502", "503", "504"])
            # Stopping if the error cannot be retried or retries are exhausted
            if not retryable or attempt == MAX_RETRIES - 1:
                print(f"Failed call: {error[:150]}")
                return None

            # Waiting before retrying using exponential backoff
            wait_time = (2 ** attempt) + random.uniform(0, 0.5)
            reason = "JSON generation failed" if json_gen_error else "Rate limit / server error"
            print(f"{reason}. Retry {attempt + 1}/{MAX_RETRIES} in {wait_time:.1f}s")
            time.sleep(wait_time)

    # Return None if all retry attempts fail
    return None

In [ ]:
import json

# Building a prompt for the LLM to verify relation triples
def build_verification_prompt(batch: list) -> str:
    numbered = []

    # Formatting each candidate with its entities and relation
    for i, r in enumerate(batch, 1):
        marked = r.get("marked_text", "")

        numbered.append( f'{i}. Marked sentence: "{marked}"\n'
            f'   Claimed head_type: "{r.get("head_type", "")}", '
            f'relation: "{r.get("relation", "")}", '
            f'tail_type: "{r.get("tail_type", "")}"')

    # Combine all candidates into one prompt
    numbered_text = "\n".join(numbered)

    # Returning the complete verification prompt
    return f"""You are a Cyber Threat Intelligence (CTI) relation verification expert.

Each "Marked sentence" has its head entity wrapped in [E1] ... [/E1] and its
tail entity wrapped in [E2] ... [/E2].

Evaluate each item independently:
1. relation_supported: Is the claimed relation between [E1] and [E2] factually supported by the sentence?
2. span_precise: Do [E1] and [E2] wrap ONLY the entity mention, nothing more or less?

Respond ONLY with a valid JSON array containing exactly {len(batch)} items, in order:
[{{"relation_supported": true/false, "span_precise": true/false, "confidence": 0.0-1.0, "reason": "short explanation"}}]

Items:
{numbered_text}
"""


# Parse the LLM verification response and add the results to each candidate
def parse_verification_batch_response(batch: list, gen_text: str) -> list:
    """Parses JSON batch response into row dicts with verifier results."""
    gen_text = gen_text.strip()
    # Extract and parse the JSON array from the response
    try:
        start_idx = gen_text.find("[")
        end_idx = gen_text.rfind("]")
        verdicts = json.loads(gen_text[start_idx : end_idx + 1])
    except Exception:
        verdicts = None

    # Handling invalid or mismatched JSON responses
    if not isinstance(verdicts, list) or len(verdicts) != len(batch):
        return [
            {
                **r,
                "verifier_relation_supported": False,
                "verifier_span_precise": False,
                "verifier_confidence": 0.0,
                "verifier_reason": f"Parse error: invalid or mismatched batch JSON | raw: {gen_text[:100]}",
                "verifier_parse_failed": True,
            }
            for r in batch
        ]

    rows = []

    # Adding the verification results to each candidate
    for r, v in zip(batch, verdicts):
        try:
            rows.append({
                **r,
                "verifier_relation_supported": bool(v.get("relation_supported", False)),
                "verifier_span_precise": bool(v.get("span_precise", False)),
                "verifier_confidence": float(v.get("confidence", 0.0)),
                "verifier_reason": str(v.get("reason", "")),
                "verifier_parse_failed": False,
            })
        except Exception as e:
            # Handling errors when parsing an individual verification result
            rows.append({
                **r,
                "verifier_relation_supported": False,
                "verifier_span_precise": False,
                "verifier_confidence": 0.0,
                "verifier_reason": f"Item parse error: {e}",
                "verifier_parse_failed": True,
            })

    return rows

In [ ]:
import os
import json
import time
import pandas as pd

# Setting the minimum semantic similarity score for pseudo-label selection
SEMANTIC_SIMILARITY_THRESHOLD = 0.80

# Updating semantic similarity values for newly added rows
def update_similarity_for_new_rows(df: pd.DataFrame, batch_size: int = 64) -> pd.DataFrame:
    if df.empty:
        return df

    # Creating the similarity column if it does not already exist
    if "semantic_similarity" not in df.columns:
        df["semantic_similarity"] = None

    # Finding rows where semantic similarity has not been calculated
    missing_mask = df["semantic_similarity"].isna()
    if not missing_mask.any():
        return df

    return df

# Loading verification progress from a JSON file
def load_progress(path: str) -> dict:
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return {"next_index": 0}


# Saving verification progress to a JSON file
def save_progress(progress: dict, path: str):
    with open(path, "w") as f:
        json.dump(progress, f, indent=2)


# Appending new verified rows to the existing verification results
def append_verified(new_rows: list, output_path: str) -> pd.DataFrame:
    new_df = pd.DataFrame(new_rows)

    # Loading existing results if the output file already exists
    if os.path.exists(output_path):
        existing_df = pd.read_json(output_path)
        combined = pd.concat(
            [existing_df, new_df],
            ignore_index=True)
    else:
        combined = new_df

    # Removing duplicate candidate relations
    if not combined.empty:
        combined = combined.drop_duplicates(
            subset=["marked_text", "relation"],
            keep="last")

    # Saving the combined verification results
    combined.to_json(
        output_path,
        orient="records",
        indent=2
    )

    return combined


# Saving  verified candidates and generate pseudo label checkpoints
def save_pseudo_label_checkpoints(
    verified_df: pd.DataFrame,
    attack_pseudo_path: str,
    cosine_pseudo_path: str,
    verified_path: str,
    verbose: bool = False,
) -> pd.DataFrame:

    # Returning if there are no verified candidates
    if verified_df.empty:
        if verbose:
            print("No verified candidates available for checkpoint.")
        return verified_df

    # Creating a copy of the verified data
    df = verified_df.copy()

    # Defining the columns required for pseudo-label selection
    required_columns = [
        "verifier_relation_supported",
        "verifier_span_precise",
        "verifier_confidence",
        "head_attack_match",
        "tail_attack_match",
    ]

    # Adding missing columns with default values
    for col in required_columns:
        if col not in df.columns:
            if col == "verifier_confidence":
                df[col] = 0.0
            else:
                df[col] = False

    # Checking whether either entity is grounded in MITRE ATT&CK
    is_attack_grounded = ( df["head_attack_match"].astype(bool) | df["tail_attack_match"].astype(bool))

    # Selecting pseudo-labels supported by the verifier and ATT&CK grounding
    valid_pseudo_labels = df[ (df["verifier_relation_supported"] == True) & ( (df["verifier_confidence"] >= 0.90) | ((df["verifier_confidence"] >= 0.75) & is_attack_grounded ))].copy()

    # Updating semantic similarity values
    df = update_similarity_for_new_rows(df, batch_size=64)

    # Checking which candidates meet the semantic similarity threshold
    is_semantically_similar = ( df["semantic_similarity"] >= SEMANTIC_SIMILARITY_THRESHOLD)

    # Selecting pseudo-labels using semantic similarity as additional evidence
    valid_pseudo_labels_cosine = df[
        (df["verifier_relation_supported"] == True) & ((df["verifier_confidence"] >= 0.90) | ( (df["verifier_confidence"] >= 0.75) &  is_semantically_similar ) )].copy()

    # Saving the complete verified results
    df.to_json( verified_path, orient="records", indent=2)

    # Saving pseudo-labels supported by ATT&CK grounding
    valid_pseudo_labels.to_json(attack_pseudo_path, orient="records", indent=2 )

    # Saving pseudo-labels supported by semantic similarity
    valid_pseudo_labels_cosine.to_json( cosine_pseudo_path, orient="records", indent=2)

    # Printing checkpoint statistics when requested
    if verbose:
        print(
            f"Checkpoint saved:"
            f"\n  Verified candidates : {len(df)}"
            f"\n  ATT&CK pseudo-labels: {len(valid_pseudo_labels)}"
            f"\n  Cosine pseudo-labels : {len(valid_pseudo_labels_cosine)}"
        )

    return df


# Running the LLM verification process in batches with progress tracking
def run_verification_session(
    df: pd.DataFrame,
    batch_size: int = 8,
    progress_path: str = "verification_progress.json",
    output_path: str = "groq_verified_full_log.json",
    attack_pseudo_path: str = "verified_pseudo_labels.json",
    cosine_pseudo_path: str = "verified_pseudo_labels_cosine.json",
    print_every: int = 100,
) -> pd.DataFrame:

    # Stopping if there are no candidates to verify
    if df.empty:
        print("No candidates to verify.")
        return df

    # Loading the previous verification progress
    progress = load_progress(progress_path)
    start_idx = progress.get("next_index", 0)
    records = df.to_dict(orient="records")
    total = len(records)

    # Checking if all candidates have already been processed
    if start_idx >= total:
        print(f"All {total} candidates already verified.")
        if os.path.exists(output_path):
            combined = pd.read_json(output_path)

            # Rebuilding the latest pseudo-label checkpoints
            combined = save_pseudo_label_checkpoints(
                verified_df=combined,
                attack_pseudo_path=attack_pseudo_path,
                cosine_pseudo_path=cosine_pseudo_path,
                verified_path=output_path,
                verbose=True,
            )
            return combined
        return pd.DataFrame()

    # Printing the starting verification position
    print(f"Verifying candidates {start_idx}/{total} (batch_size={batch_size})")

    # Recording the verification start time
    start_time = time.time()
    idx = start_idx
    last_printed_count = (start_idx // print_every) * print_every

    # Processing candidates in batches
    while idx < total:
        batch = records[idx : idx + batch_size]

        # Building the verification prompt and send it to the LLM
        prompt = build_verification_prompt(batch)
        response = call_groq(prompt, max_tokens=400 * len(batch))

        # Stopping and save progress if the daily API limit is reached
        if response == "DAILY_LIMIT_EXCEEDED":
            print(f"\nStopped at {idx}/{total} — quota reached.")
            print("Run this cell again next session to resume.")
            break

        # Extracting the generated verification response
        gen_text = ( response.choices[0].message.content if response else "" )

        # Parsing the verification results
        rows = parse_verification_batch_response(batch, gen_text)
        idx += len(batch)

        # Saving the newly verified rows
        combined = append_verified(rows, output_path)

        # Updating semantic similarity values
        combined = update_similarity_for_new_rows(combined, batch_size=64)

        # Printing checkpoint summary every 100 candidates
        should_print = (idx - last_printed_count >= print_every) or (idx >= total)

        # Saving verified results and pseudo-label checkpoints
        combined = save_pseudo_label_checkpoints(
            verified_df=combined,
            attack_pseudo_path=attack_pseudo_path,
            cosine_pseudo_path=cosine_pseudo_path,
            verified_path=output_path,
            verbose=should_print,
        )

        # Saving the current verification position
        progress["next_index"] = idx
        save_progress(progress, progress_path)

        # Printing progress at the configured interval
        if should_print:
            print(f"Verified {idx}/{total}")
            last_printed_count = (idx // print_every) * print_every

    # Calculating and printing the total verification time
    elapsed = time.time() - start_time
    print(f"\nCompleted/Paused in {elapsed:.1f}s.")

    # Loading the final verified results
    combined = ( pd.read_json(output_path) if os.path.exists(output_path) else pd.DataFrame())

    # Printitng verification statistics
    if not combined.empty and "verifier_relation_supported" in combined.columns:
        n_relation_ok = int(combined["verifier_relation_supported"].sum())
        n_span_ok = int(combined["verifier_span_precise"].sum())
        print(f"relation_supported = True : {n_relation_ok}/{len(combined)}")
        print(f"span_precise = True : {n_span_ok}/{len(combined)}")

    return combined

In [ ]:
# Runing the verification pipeline on the raw LLM candidate triples
verified_df = run_verification_session(
    raw_candidates_df,
    batch_size=8,
    progress_path=PROGRESS_PATH,
    output_path=VERIFIED_PATH,
    attack_pseudo_path=ATTACK_PSEUDO_PATH,
    cosine_pseudo_path=COSINE_PSEUDO_PATH,
)

# Printing the first few verified candidates
verified_df.head()

Verifying candidates 3016/3979 (batch_size=8)...
Checkpoint saved:
  Verified candidates : 3104
  ATT&CK pseudo-labels: 1823
  Cosine pseudo-labels : 1763
Verified 3104/3979
Rate limit / server error. Retry 1/5 in 1.4s...
Checkpoint saved:
  Verified candidates : 3200
  ATT&CK pseudo-labels: 1868
  Cosine pseudo-labels : 1806
Verified 3200/3979
Rate limit / server error. Retry 1/5 in 1.0s...
Checkpoint saved:
  Verified candidates : 3304
  ATT&CK pseudo-labels: 1941
  Cosine pseudo-labels : 1878
Verified 3304/3979
Checkpoint saved:
  Verified candidates : 3400
  ATT&CK pseudo-labels: 1991
  Cosine pseudo-labels : 1924
Verified 3400/3979
Checkpoint saved:
  Verified candidates : 3504
  ATT&CK pseudo-labels: 2043
  Cosine pseudo-labels : 1976
Verified 3504/3979
Checkpoint saved:
  Verified candidates : 3600
  ATT&CK pseudo-labels: 2112
  Cosine pseudo-labels : 2045
Verified 3600/3979
Checkpoint saved:
  Verified candidates : 3704
  ATT&CK pseudo-labels: 2160
  Cosine pseudo-labels : 2093

,marked_text,head_type,tail_type,relation,head,tail,sentence,source_file,head_attack_match,head_attack_name,head_attack_id,tail_attack_match,tail_attack_name,tail_attack_id,verifier_relation_supported,verifier_span_precise,verifier_confidence,verifier_reason,verifier_parse_failed,semantic_similarity
0,related report : https://www.securityartwork.e...,FILEPATH,hash,indicates,Ukraine_election_2019_polls.doc,8a35b6ecdf43f42dbf1e77235d6017faa70d9c68930bdc...,related report : https://www.securityartwork.e...,apt_subset/APT28__IOC__2019-04-09-ioc-mark.txt,False,None,None,False,None,None,True,True,0.95,"The sentence links the file name to its hash, ...",False,NaN
1,related report : https://www.securityartwork.e...,FILEPATH,url,communicates-with,Ukraine_election_2019_polls.doc,functiondiscovery[.,related report : https://www.securityartwork.e...,apt_subset/APT28__IOC__2019-04-09-ioc-mark.txt,False,None,None,False,None,None,False,False,0.90,"The tail entity is ""functiondiscovery[."" which...",False,NaN
2,]182 [E1] related doc download.doc [/E1] [E2] ...,FILEPATH,hash,indicates,related doc download.doc,8cccdce85beca7b7dc805a7f048fcd1bc8f7614dd7e13c...,]182 related doc download.doc 8cccdce85beca7b7...,apt_subset/APT28__IOC__2019-04-09-ioc-mark.txt,False,None,None,False,None,None,True,False,0.85,"The file is associated with the hash, supporti...",False,NaN
3,]182 [E1] related doc download.doc [/E1] 8cccd...,FILEPATH,url,communicates-with,related doc download.doc,http://beatguitar.com/,]182 related doc download.doc 8cccdce85beca7b7...,apt_subset/APT28__IOC__2019-04-09-ioc-mark.txt,False,None,None,False,None,None,True,False,0.85,"The file is said to communicate with the URL, ...",False,NaN
4,Internal functions were added to exports (auth...,tools,threat-actor,authored-by,SysWhispers3,the adversary,Internal functions were added to exports (auth...,apt_subset/APT29__CERT.PL__IoC_Reference_.pdf,False,None,None,False,None,None,False,False,0.90,The sentence states that SysWhispers3 is a lib...,False,NaN


In [ ]:
for col in ["verifier_relation_supported", "verifier_span_precise", "verifier_confidence",
            "head_attack_match", "tail_attack_match"]:
    if col not in verified_df.columns:
        verified_df[col] = False if col != "verifier_confidence" else 0.0

# Checking whether at least one entity is grounded in MITRE ATT&CK
is_attack_grounded = ( verified_df["head_attack_match"].astype(bool) | verified_df["tail_attack_match"].astype(bool))

# Selecting candidates that meet the verifier confidence and ATT&CK grounding criteria
valid_pseudo_labels = verified_df[(verified_df["verifier_relation_supported"] == True) & ((verified_df["verifier_confidence"] >= 0.90) | ((verified_df["verifier_confidence"] >= 0.75) & is_attack_grounded))].copy()

# Counting accepted candidates with imprecise entity spans
n_span_imprecise_in_accepted = int((~valid_pseudo_labels["verifier_span_precise"]).sum())

# Printing the verification summary
print("--- Verification Summary (LLM verifier + ATT&CK grounding) ---")

print(f"Total Candidates Input   : {len(verified_df)}")
print(f"ATT&CK-Grounded Candidates : {int(is_attack_grounded.sum())}")
print(f"Validated Pseudo-Labels : {len(valid_pseudo_labels)}")
print(f"Filtered Out (Noise) : {len(verified_df) - len(valid_pseudo_labels)}")
print(f"  (of which span_precise=False: {n_span_imprecise_in_accepted} - worth a manual look, "
      f"but not auto-rejected)")

# Printitng the location of the saved pseudo-label checkpoint
print(
    f"\nCheckpoint file:"
    f"\n{ATTACK_PSEUDO_PATH}"
)

# Confirm that the pseudo-label file has been saved
print("\nSaved verified_pseudo_labels.json")

--- Verification Summary (LLM verifier + ATT&CK grounding) ---
Total Candidates Input        : 3728
ATT&CK-Grounded Candidates    : 935
Validated Pseudo-Labels       : 2179
Filtered Out (Noise)          : 1549
  (of which span_precise=False: 181 -- worth a manual look, but not auto-rejected)

Checkpoint file:
/content/drive/MyDrive/apt_extraction/verified_pseudo_labels.json

Saved verified_pseudo_labels.json


In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util as st_util

# Defining the model and threshold used for semantic similarity
SIMILARITY_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
SEMANTIC_SIMILARITY_THRESHOLD = 0.5

# Loading the sentence similarity model
print(f"Loading similarity model: {SIMILARITY_MODEL_ID}")
similarity_model = SentenceTransformer(SIMILARITY_MODEL_ID)
print("Similarity model loaded.")
print(f"Semantic similarity threshold: {SEMANTIC_SIMILARITY_THRESHOLD}")


# Converting a relation triple into a readable statement
def triple_to_statement(head: str, relation: str, tail: str) -> str:
    readable_relation = str(relation).replace("-", " ")
    return f"{head} {readable_relation} {tail}"


# Computing semantic similarity between sentences and relation statements
def compute_semantic_similarity( df: pd.DataFrame, batch_size: int = 64) -> pd.DataFrame:
    df = df.copy()

    # Returning an empty similarity column if there are no rows
    if df.empty:
        df["semantic_similarity"] = pd.Series(dtype=float)
        return df

    # Selecting the original sentence or marked sentence for comparison
    if "sentence" in df.columns:
        sentences = df["sentence"].fillna("").astype(str).tolist()
    else:
        sentences = df["marked_text"].fillna("").astype(str).tolist()

    # Converting each extracted triple into a readable statement
    statements = [triple_to_statement(h, r, t) for h, r, t in zip(df["head"], df["relation"], df["tail"])]

    # Generating embeddings for the original sentences
    sent_emb = similarity_model.encode(
        sentences,
        batch_size=batch_size,
        convert_to_tensor=True,
        show_progress_bar=False,
    )

    # Generating embeddings for the relation statements
    stmt_emb = similarity_model.encode(
        statements,
        batch_size=batch_size,
        convert_to_tensor=True,
        show_progress_bar=False,
    )

    # Calculating cosine similarity between each sentence and its statement
    sims = (st_util.cos_sim(sent_emb, stmt_emb).diagonal().cpu().numpy())

    # Storing the similarity scores in the DataFrame
    df["semantic_similarity"] = sims
    return df


# Calculating semantic similarity only for newly added candidates
def update_similarity_for_new_rows( df: pd.DataFrame, batch_size: int = 64) -> pd.DataFrame:
    df = df.copy()

    # Creating the similarity column if it does not already exist
    if "semantic_similarity" not in df.columns:
        df["semantic_similarity"] = np.nan

    # Identifying rows that do not yet have a similarity score
    missing_mask = df["semantic_similarity"].isna()

    # Skipping calculation if all rows already have similarity scores
    if not missing_mask.any():
        print("Semantic similarity already exists for all verified candidates.")
        return df

    # Selecting candidates that need similarity scores
    new_rows = df.loc[missing_mask].copy()
    print(f"Computing semantic similarity for {len(new_rows)} new verified candidates.")

    # Calculating similarity scores for the new candidates
    new_rows = compute_semantic_similarity(new_rows, batch_size=batch_size)

    # Adding the new similarity scores back to the main DataFrame
    df.loc[missing_mask, "semantic_similarity"] = new_rows["semantic_similarity"].values

    print("Semantic similarity update complete.")
    return df

Loading similarity model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Similarity model loaded.
Semantic similarity threshold: 0.5


In [ ]:
# Identifying accepted pseudo-labels with low semantic similarity
low_sim = valid_pseudo_labels[
    valid_pseudo_labels["semantic_similarity"] < SEMANTIC_SIMILARITY_THRESHOLD
]

# Printing the semantic similarity summary
print("--- Semantic Similarity Summary (independent of ATT&CK grounding) ---")

print(f"Accepted pseudo-labels     : {len(valid_pseudo_labels)}")
print(f"Mean semantic similarity   : {valid_pseudo_labels['semantic_similarity'].mean():.3f}")
print(f"Median semantic similarity : {valid_pseudo_labels['semantic_similarity'].median():.3f}")

# Pritning the number and percentage of labels below the similarity threshold
print(f"Below threshold ({SEMANTIC_SIMILARITY_THRESHOLD})       : {len(low_sim)} "
      f"({100 * len(low_sim) / max(len(valid_pseudo_labels), 1):.1f}%)")

# Printing the lowest scoring accepted triples for manual inspection
if not low_sim.empty:
    print("\nLowest-similarity accepted triples (worth a manual look):")
    print(
        low_sim.sort_values("semantic_similarity")
        [["marked_text", "relation", "semantic_similarity"]]
        .head(10)
        .to_string(index=False)
    )

--- Semantic Similarity Summary (independent of ATT&CK grounding) ---
Accepted pseudo-labels     : 2179
Mean semantic similarity   : nan
Median semantic similarity : nan
Below threshold (0.5)       : 0 (0.0%)


In [ ]:
# Identifying candidates that meet the semantic similarity threshold
is_semantically_similar = verified_df["semantic_similarity"] >= SEMANTIC_SIMILARITY_THRESHOLD

# Selecting pseudo labels supported by the verifier and semantic similarity
valid_pseudo_labels_cosine = verified_df[(verified_df["verifier_relation_supported"] == True) & ((verified_df["verifier_confidence"] >= 0.90) | ((verified_df["verifier_confidence"] >= 0.75) & is_semantically_similar))].copy()

# Printing the verification summary
print("--- Verification Summary (LLM verifier + cosine similarity) ---")

print(f"Total Candidates Input : {len(verified_df)}")
print(f"Semantically-Similar Candidates : {int(is_semantically_similar.sum())}")
print(f"Validated Pseudo-Labels  : {len(valid_pseudo_labels_cosine)}")
print(f"Filtered Out (Noise) : {len(verified_df) - len(valid_pseudo_labels_cosine)}")

# Saving the validated pseudo-labels to a JSON file
valid_pseudo_labels_cosine.to_json(
    COSINE_PSEUDO_PATH,
    orient="records",
    indent=2
)
print(f"\nSaved {COSINE_PSEUDO_PATH}")

--- Verification Summary (LLM verifier + cosine similarity) ---
Total Candidates Input          : 3728
Semantically-Similar Candidates : 0
Validated Pseudo-Labels         : 2112
Filtered Out (Noise)            : 1616

Saved /content/drive/MyDrive/apt_extraction/verified_pseudo_labels_cosine.json


In [ ]:
# Creating a unique key using the marked sentence and relation
def add_key(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["_key"] = list(zip(df["marked_text"], df["relation"]))
    return df

# Adding unique keys to both pseudo-label datasets
attack_df = add_key(valid_pseudo_labels)
cosine_df = add_key(valid_pseudo_labels_cosine)

# Creating sets of accepted candidate keys
attack_keys = set(attack_df["_key"])
cosine_keys = set(cosine_df["_key"])

# Comparing the candidates accepted by both pipelines
both_keys = attack_keys & cosine_keys
only_attack_keys = attack_keys - cosine_keys
only_cosine_keys = cosine_keys - attack_keys
union_keys = attack_keys | cosine_keys

# Printing the acceptance comparison summary
print("--- ATT&CK-based vs Cosine-based Acceptance Comparison ---")

print(f"Accepted via ATT&CK pipeline  : {len(attack_keys)}")
print(f"Accepted via cosine pipeline : {len(cosine_keys)}")
print(f"Accepted by BOTH pipelines : {len(both_keys)}")
print(f"Only accepted via ATT&CK : {len(only_attack_keys)}")
print(f"Only accepted via cosine : {len(only_cosine_keys)}")

# Calculating the Jaccard overlap between the two pipelines
if union_keys:
    print(f"Jaccard overlap (both/union) : {len(both_keys) / len(union_keys):.3f}")

# Selecting candidates accepted by only one of the two pipelines
only_attack_df = attack_df[attack_df["_key"].isin(only_attack_keys)]
only_cosine_df = cosine_df[cosine_df["_key"].isin(only_cosine_keys)]

# Printing examples accepted only through ATT&CK grounding
if not only_attack_df.empty:
    print("\nSample triples accepted ONLY via ATT&CK grounding (rejected by cosine):")
    print( only_attack_df[["marked_text", "relation", "semantic_similarity"]].head(10) .to_string(index=False))

# Printing examples accepted only through cosine similarity
if not only_cosine_df.empty:
    print("\nSample triples accepted ONLY via cosine similarity (rejected by ATT&CK):")
    print(only_cosine_df[["marked_text", "relation", "head_attack_match", "tail_attack_match"]].head(10).to_string(index=False))

# Saving the disagreement cases
# Save candidates accepted only through the ATT&CK pipeline
only_attack_df.drop(columns="_key").to_json("disagreement_attack_only.json", orient="records", indent=2)
only_cosine_df.drop(columns="_key").to_json("disagreement_attack_only.json", orient="records", indent=2)

print("\nSaved disagreement_attack_only.json and disagreement_cosine_only.json for manual review.")

--- ATT&CK-based vs Cosine-based Acceptance Comparison ---
Accepted via ATT&CK pipeline  : 2179
Accepted via cosine pipeline  : 2112
Accepted by BOTH pipelines    : 2112
Only accepted via ATT&CK      : 67
Only accepted via cosine      : 0
Jaccard overlap (both/union)  : 0.969

Sample triples accepted ONLY via ATT&CK grounding (rejected by cosine):
                                                                                                                                                                                                                                                                             marked_text          relation  semantic_similarity
                                                                                                                                                                                              ]com/BMW.html URL [E1] ENVYSCOUT [/E1] delivering [E2] SNOWYAMBER [/E2] ISO badriatimimi[.          delivers                  NaN
          

In [ ]:
# Printing overall candidate and relation statistics
print("\n========== Statistics ==========")

print(f"Total candidates: {len(raw_candidates_df)}")

# Printing relation statistics if extracted candidates are available
if not raw_candidates_df.empty and "relation" in raw_candidates_df.columns:

    # Counting the number of unique relation types
    print(f"Unique relations: {raw_candidates_df['relation'].nunique()}")

    # Pritnitng the distribution of accepted relations
    print("\nAccepted relation distribution:")
    print(
        valid_pseudo_labels["relation"].value_counts()
        if not valid_pseudo_labels.empty
        else "None"
    )

    # Printing the distribution of rejected relations
    print("\nRejected relation distribution:")
    print(verified_df[verified_df["verifier_relation_supported"] == False]["relation"].value_counts() if not verified_df.empty else "None")

else:
    # printitnh a message when no valid triples are available
    print("No valid extracted triples to show statistics for.")


========== Statistics ==========
Total candidates: 3979
Unique relations: 24

Accepted relation distribution:
relation
uses                 590
targets              440
related-to           140
communicates-with    117
exploits              94
indicates             85
located-at            71
attributed-to         68
drops                 67
hosts                 63
delivers              56
compromises           51
impersonates          51
downloads             46
exfiltrates-to        42
authored-by           42
consists-of           40
duplicate-of          26
based-on              25
controls              17
originates-from       15
owns                  13
variant-of            10
beacons-to            10
Name: count, dtype: int64

Rejected relation distribution:
relation
targets              206
uses                 148
indicates             90
hosts                 78
communicates-with     73
related-to            70
drops                 64
located-at            54
authored-by 

In [ ]:
# Selecting validated pseudo-labels or fall back to the raw candidates
source_df = ( valid_pseudo_labels  if "valid_pseudo_labels" in globals() and not valid_pseudo_labels.empty  else raw_candidates_df)

# Converting the selected data into the labelled dataset format
llm_labelled_df = source_df[["marked_text", "head_type", "tail_type", "relation"]].reset_index(drop=True)

# Printitng the distribution of relations in the converted dataset
print("Relation distribution in converted set:")
print(llm_labelled_df["relation"].value_counts().to_string())

# Saving the converted labelled dataset as a JSON file
llm_labelled_df.to_json("llm_extracted_labelled_format.json", orient="records", indent=2)
print("\nSaved llm_extracted_labelled_format.json - same schema as `all_df` in Baseline_Training.ipynb")

Relation distribution in converted set:
relation
uses                 590
targets              440
related-to           140
communicates-with    117
exploits              94
indicates             85
located-at            71
attributed-to         68
drops                 67
hosts                 63
delivers              56
compromises           51
impersonates          51
downloads             46
exfiltrates-to        42
authored-by           42
consists-of           40
duplicate-of          26
based-on              25
controls              17
originates-from       15
owns                  13
variant-of            10
beacons-to            10

Saved llm_extracted_labelled_format.json -- same schema as `all_df` in Baseline_Training.ipynb


In [ ]:
from google.colab import files
import os

files_to_download = [
    ATTACK_PSEUDO_PATH,
    COSINE_PSEUDO_PATH,
    os.path.join(DRIVE_DIR, "llm_extracted_labelled_format.json")
]

for file_path in files_to_download:
    if os.path.exists(file_path):
        files.download(file_path)
    else:
        print(f"File not found, skipped: {file_path}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File not found, skipped: /content/drive/MyDrive/apt_extraction/llm_extracted_labelled_format.json
